# NB3 · Temel model ve dürüst değerlendirme
### Baseline and honest evaluation

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr  
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

---

Bu defter atölyenin dönüm noktasıdır. Model kurmak on beş satır sürmektedir;
defterin geri kalanı o modelin ne kadar işe yaradığını dürüstçe ölçmeye ayrılmıştır.
Oran bilinçlidir ve mesleki pratikteki oranı yansıtmaktadır.


## 0. Kurulum · Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['mimic_web.py', 'evaluate.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import mimic_web as mw
import evaluate as ev

cohort = mw.build_cohort()


## 1. Ayrım: Hasta düzeyinde, yatış düzeyinde değil
### The split: by patient, never by stay

Bu kohortta bir hastanın birden fazla yoğun bakım yatışı bulunabilmektedir. Veriyi
satır düzeyinde rastgele ayırırsanız aynı hastanın bir yatışı eğitim kümesine, diğeri
test kümesine düşer. Model o hastayı tanıdığı için test başarımı yükselir ve bu
yükselme gerçek değildir.

Bu, üretken yapay zekâ araçlarının en sık yaptığı sessiz hatalardan biridir. `train_test_split`
istendiğinde varsayılan olarak satır düzeyinde ayırmaktadır ve kimse aksini söylemedikçe
hasta kimliğini dikkate almaz. Hücrede `GroupShuffleSplit` kullanılmasının sebebi budur.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

features = mw.feature_columns(cohort)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(
    splitter.split(cohort, cohort['prolonged_stay'], groups=cohort['subject_id'])
)
train, test = cohort.iloc[train_idx], cohort.iloc[test_idx]

overlap = set(train['subject_id']) & set(test['subject_id'])
print(f'Eğitim yatışı: {len(train)}   Test yatışı: {len(test)}')
print(f'İki kümede birden görünen hasta sayısı: {len(overlap)}')
print(f'Eğitim sonuç oranı: {train["prolonged_stay"].mean():.1%}')
print(f'Test sonuç oranı:   {test["prolonged_stay"].mean():.1%}')


Son iki satır önemlidir. İki kümedeki sonuç oranı birbirinden belirgin biçimde
farklıysa, bu kadar küçük bir kümede tek bir ayrımın sonuçları ne kadar kaydırabildiğini
görüyorsunuz demektir. Böyle bir durumda tek bir ayrım üzerinden rapor edilen her sayı,
ayrımın kendisine ait bir tesadüf olabilir.


## 2. Ön işleme zinciri · The preprocessing pipeline

Ölçekleyici, tamamlayıcı ve kodlayıcı eğitim kümesi üzerinde öğrenilmekte, test
kümesine yalnızca uygulanmaktadır. Bunların tamamı tek bir `Pipeline` içindedir.

Neden pipeline: Tamamlayıcıyı ayrımdan önce tüm veri üzerinde çalıştırmak, test
kümesindeki değerlerin ortancasını eğitim verisine taşımaktadır. Bu, sessiz sızıntıdır.
Hiçbir uyarı vermez, başarımı yükseltir ve kod okunduğunda doğru görünür. Zincirin
tamamını pipeline içine koymak bu hatayı yapısal olarak imkânsız kılmaktadır.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric = cohort[features].select_dtypes(include='number').columns.tolist()
categorical = [c for c in features if c not in numeric]

blocks = []
if numeric:
    blocks.append(('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric))
if categorical:
    blocks.append(('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical))

model = Pipeline([
    ('prepare', ColumnTransformer(blocks)),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

model.fit(train[features], train['prolonged_stay'])
print(f'Sayısal öznitelik: {len(numeric)}   Kategorik öznitelik: {len(categorical)}')
print('Model eğitildi.')


Temel model olarak lojistik regresyon seçilmiştir. Sebebi başarımı değil, üç
özelliğidir: Katsayıları okunabilir, olasılık üretir ve az veriyle aşırı uyum yapma
eğilimi düşüktür. Temel model iyi olmak için değil, aşılmak için vardır. Bir gradyan
artırma modeli buradan daha iyi bir sayı üretirse, o sayının bu kohortta anlamlı olup
olmadığını yine aşağıdaki güven aralığına bakarak karar vereceksiniz.


## 3. Doğruluk tuzağı · The accuracy trap

Aşağıdaki hücre yalnızca doğruluk yazdırmaktadır. Rakamı okuyunuz, sonra bir sonraki
hücreyi çalıştırınız.


In [ ]:
accuracy = model.score(test[features], test['prolonged_stay'])
print(f'Test doğruluğu: {accuracy:.1%}')


In [ ]:
null = ev.null_comparison(test['prolonged_stay'])
print(f"Hiçbir şey öğrenmeyen model ({null['strategy']}): {null['accuracy']:.1%}")
print()
print('Modelin doğruluğu ile her zaman çoğunluk sınıfını söyleyen bir kuralın')
print('doğruluğu arasındaki farkı okuyunuz. Bu fark, modelin gerçekten kattığı değerdir.')


Derste anlatılan nokta buydu. Yüzde 7 prevalansta her zaman olumsuz sınıfı söyleyen
bir kural yüzde 93 doğruluk vermektedir. Doğruluk, dengesiz bir klinik problemde
modelin ne yaptığını değil, hastalığın ne kadar nadir olduğunu ölçmektedir.


## 4. Dürüst değerlendirme raporu · The honest evaluation report

Aşağıdaki çağrı, dersteki altı maddelik listeyi uygulamaktadır: Güven aralığıyla ayrım
gücü, kalibrasyon, çalışma noktası, klinik karşılık, alt grup dökümü ve boş karşılaştırma.

`target_sensitivity` parametresi kartınızdaki altıncı sorunun cevabıdır. Kaçırma daha
pahalıysa yüksek tutulur; yanlış alarm daha pahalıysa düşürülür ve eşik özgüllükten
seçilir. Bu parametre klinik bir karardır, teknik bir varsayılan değildir.


In [ ]:
report = ev.honest_report(
    test['prolonged_stay'],
    model.predict_proba(test[features])[:, 1],
    groups=test['gender'],
    target_sensitivity=0.80,
    label='temel lojistik regresyon · yoğun bakımda uzamış kalış',
)


In [ ]:
fig = ev.plot_curves(
    test['prolonged_stay'],
    model.predict_proba(test[features])[:, 1],
)


## 5. Raporun okunması · Reading the report

Sırasıyla şu dört noktaya bakınız.

**Güven aralığı.** Aralık 0,5 değerini içeriyorsa model şanstan ayırt edilememektedir.
Nokta tahmini ne olursa olsun bu böyledir. Küçük kohortlarda nokta tahminine bakmak,
aralığı görmezden gelmek demektir.

**Kalibrasyon eğimi.** Birin altındaysa model aşırı güvenlidir: Yüksek olasılıkları
olduğundan yüksek, düşükleri olduğundan düşük vermektedir. Eşik belirleyen bir
klinisyen için bu, eşiğin sandığından farklı bir yerde durması anlamına gelmektedir.

**Klinik karşılık.** Her yüz hastada kaç uyarı çıktığına ve kaçının doğru olduğuna
bakınız. Bu satır, dersteki Epic Sepsis Model tahtasının bu modeldeki karşılığıdır.

**Alt grup dökümü.** Bazı alt gruplar için sayı yerine *insufficient sample*
yazmaktadır. Bu bir eksiklik değil, bir bulgudur: Hiç test edilmemiş bir grup için
sistemin adil olduğu gösterilemez. Bu ifadeyi bir model kartına yazmak, o hücreye
uydurma bir sayı yazmaktan daha dürüsttür.


### Bu senaryoda beklenen sonuç

Rapor büyük olasılıkla olumsuz çıkacaktır ve çıkması beklenmektedir. Yüz hastalık bir
kümede güven aralığı geniştir, alt grup dökümü çoğunlukla boştur ve modelin boş
karşılaştırmaya üstünlüğü küçüktür.

Derste anlatılan Epic Sepsis Model, 38.455 yatış üzerinde dış doğrulandığında 0,63
eğri altı alan vermişti. Buradaki veri onun binde birinden küçüktür. Dördüncü istemin
sonunda soracağınız *DEVREYE ALIR MIYDIM* sorusunun cevabı bu senaryoda hayırdır.

Bunu kendi kurduğunuz sistemde görmek, bir başkasının başarısızlığını dinlemekten
farklıdır. Aynı şüpheyle kendi probleminize geçmeniz beklenmektedir.


## 6. İstemlerle devam · Continuing with the prompts

Şimdi aynı işi üretken yapay zekâya yaptırınız. Aşağıdaki bağlam bloğunu asistanınıza
veriniz, ardından istem kütüphanesindeki üçüncü ve dördüncü istemleri çalıştırınız.

### Türkçe bağlam bloğu

```
Elimde `cohort` adında bir pandas veri çerçevesi var. MIMIC-IV demo veri kümesinden
kuruldu, yüz hastadan oluşuyor ve her satır bir yoğun bakım yatışını temsil ediyor.

Sonuç sütunu: prolonged_stay (1 = yoğun bakım yatışı üç günden uzun)
Kimlik sütunları: subject_id, hadm_id, stay_id, intime
Öznitelikler: demografik bilgi, yatış bağlamı ve yatıştan sonraki ilk altı saate ait
vital ile laboratuvar değerlerinin min, max, mean ve sayım özetleri.

Karar anı yoğun bakıma kabulden altı saat sonrasıdır. Bir hastanın birden fazla
yatışı olabilir.

Kohort yüz hastadan oluşmaktadır. Bu kadar küçük bir kümede elde edilen hiçbir
başarım değerini olduğundan güçlü sunma.
```

### English context block

```
I have a pandas dataframe called `cohort`, built from the MIMIC-IV demo dataset. It
covers one hundred patients and each row is one ICU stay.

Outcome column: prolonged_stay (1 = ICU length of stay longer than three days)
Identifier columns: subject_id, hadm_id, stay_id, intime
Features: demographics, admission context, and min, max, mean and count summaries of
vital signs and laboratory results from the first six hours after ICU admission.

The decision point is six hours after ICU admission. One patient may have more than
one stay.

The cohort contains one hundred patients. Do not present any performance figure from
a cohort this small as stronger than it is.
```


## 7. Dil karşılaştırma egzersizi · Language comparison exercise

Bu adım atlanmamalıdır. Atölyenin okuryazarlık omurgasıdır.

1. Üçüncü istemi **önce Türkçe** çalıştırınız. Çıktıyı kaydediniz.
2. **Yeni bir konuşma açınız** ve aynı istemi İngilizce çalıştırınız. Yeni konuşma
   şarttır; aynı konuşmada devam ederseniz model ilk cevabını tekrarlar ve
   karşılaştırma anlamsızlaşır.
3. İki çıktıyı aşağıdaki tabloya göre karşılaştırınız.
4. Her iki çıktıya da sıfırıncı istemi, yani sorgulama istemini uygulayınız.

| Karşılaştırma başlığı | Türkçe çıktı | İngilizce çıktı |
|---|---|---|
| Seçilen model ve gerekçesi | | |
| Ayrımı hasta düzeyinde mi yaptı | | |
| Ön işlemeyi pipeline içine aldı mı | | |
| Uydurulmuş sayı var mı | | |
| Klinik terimleri doğru eşledi mi | | |
| Doğruluğu başlık sayı olarak sundu mu | | |

Bulgunuz probleminize göre değişecektir ve tek bir doğru cevabı yoktur. Kayda
geçirdiğiniz fark, atölyeden çıkardığınız en taşınabilir çıktıdır.


---

## Ayrışma noktası · Divergence point

Buraya kadar olan her şey ortak senaryo üzerinde çalıştı. Bundan sonrası sizin
probleminizdir.

`templates/cdss-canvas.md` dosyasında doldurduğunuz yedi cevabı alınız, yeni bir
konuşma açınız ve birinci istemden dördüncü isteme kadar olan zinciri baştan
çalıştırınız. Zincir değişmez, problem değişir.

Verisi olmayan katılımcılar ikinci istemin 2a sürümüyle kendi problemlerine uygun
sentetik kohort üretmektedir. Görüntü, zaman serisi veya metin ile çalışacak
katılımcılar 2b, 2c ve 2d sürümlerini kullanmaktadır.

---

### Uyarı

Bu defterde üretilen hiçbir model doğrulanmış bir klinik araç değildir. MIMIC-IV demo
verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmemektedir. Buradaki çıktılar öğretim amaçlıdır.
